# Flytris TPU scaling probe

Before running: choose **Runtime > Change runtime type > TPU**. Then run the cells from top to bottom. The first cell asks for `Flytris-colab-probe.zip` only when it is not already uploaded.

In [ ]:
# 1. Upload and extract the Flytris probe bundle.
from pathlib import Path
import os
import shutil

bundle = Path('/content/Flytris-colab-probe.zip')
if not bundle.exists():
    from google.colab import files
    print('Choose Flytris-colab-probe.zip in the upload window.')
    uploaded = files.upload()
    if 'Flytris-colab-probe.zip' not in uploaded:
        raise FileNotFoundError('Please upload the file named Flytris-colab-probe.zip.')

expected_bytes = 90_247_435
actual_bytes = bundle.stat().st_size
if actual_bytes != expected_bytes:
    raise RuntimeError(
        f'Upload is incomplete: {actual_bytes:,} of {expected_bytes:,} bytes. '
        'Delete the uploaded ZIP from the Files panel and run this cell again.'
    )

project = Path('/content/Flytris')
if project.exists():
    shutil.rmtree(project)
shutil.unpack_archive(bundle, project)
os.chdir(project)
print(f'Ready: {project} ({actual_bytes / 1_000_000:.1f} MB bundle)')

In [ ]:
# 2. Required correctness and speed test (batch 8).
import os
os.environ['PJRT_DEVICE'] = 'TPU'

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
except ImportError as exc:
    raise RuntimeError(
        'This is not a TPU runtime. Choose Runtime > Change runtime type > TPU, '
        'reconnect, and run again.'
    ) from exc

print('TPU device:', xm.xla_device())
!python scripts/tpu_probe.py --device tpu --batch 8 --steps 10

In [ ]:
# 3. Scaling tests. Run this only after cell 2 prints both PASS lines.
!python scripts/tpu_probe.py --device tpu --batch 32 --steps 10
!python scripts/tpu_probe.py --device tpu --batch 64 --steps 10

## What to send back

Copy the output from cells 2 and 3. Success means each run ends with `scatter parity: PASS` and `repeatability: PASS`. This probe benchmarks the real MaleCNS recurrent step; it does not run full Flytris training yet.